# Personality Atlas Embedding Projector

Interactive visualization of 6,694 personality trait embeddings across the 44-model atlas.

**Data source:** GitHub `Wildertrek/survey/Embeddings`  
**Embeddings:** OpenAI text-embedding-3-small (1536-dim)  
**Models:** OCEAN, HEXACO, NPI, BDI, GAD-7, MCMI, and 38 others

Run this notebook in Google Colab with no setup required (all dependencies auto-installed).

## Section 1: Setup & Install

In [ ]:
!pip install -q plotly scikit-learn umap-learn datasets huggingface_hub numpy pandas

## Section 2: Load Embeddings from GitHub

In [ ]:
import numpy as np
import pandas as pd
import requests
import io
import json

print("📥 Loading personality atlas (1536-dim embeddings from GitHub)...")

MODELS_URL = "https://raw.githubusercontent.com/Wildertrek/survey/main"
embeddings_base = f"{MODELS_URL}/Embeddings"

all_embeddings = []
model_indices = {}

for model_dir in sorted(['aam', 'bdi', 'bt', 'cest', 'cmoa', 'cs', 'disc', 'dt4', 'dtm', 'em', 'epm', 
                         'ffni', 'ffni_sf', 'fsls', 'ftm', 'gad7', 'hex', 'hsns', 'ipn', 'mbti', 
                         'mcmi', 'mcmin', 'mmpi', 'mst', 'narq', 'npi', 'ocean', 'papc', 'pct', 
                         'pni', 'rft', 'riasec', 'rit', 'scid', 'scm', 'sdt', 'sixteenpf', 'stbv', 
                         'tat', 'tci', 'tei', 'tki', 'tmp', 'wais']):
    
    try:
        emb_url = f"{embeddings_base}/{model_dir}_embeddings.csv"
        response = requests.get(emb_url, timeout=10)
        if response.status_code == 200:
            emb_df = pd.read_csv(io.StringIO(response.text))
            
            # Extract embeddings from JSON column
            embeddings = []
            for emb_json in emb_df['Embedding']:
                emb = json.loads(emb_json)
                embeddings.append(emb)
            
            embeddings_array = np.array(embeddings, dtype=np.float32)
            all_embeddings.append(embeddings_array)
            model_indices[model_dir.upper()] = (len(all_embeddings) - 1, len(embeddings_array))
            print(f"  ✅ {model_dir.upper():15s}: {len(embeddings):4d} × {len(embeddings[0])}-dim")
    except Exception as e:
        print(f"  ⚠️ {model_dir.upper()}: {str(e)[:50]}")

print(f"\n✅ Loaded {len(all_embeddings)} models")

# Combine all embeddings
X = np.vstack(all_embeddings)
print(f"✅ Combined shape: {X.shape}")

# Create metadata
df = []
CATEGORY_MAP = {
    # Trait-Based
    "OCEAN": "Trait-Based", "MBTI": "Trait-Based", "HEX": "Trait-Based",
    "EPM": "Trait-Based", "SIXTEENPF": "Trait-Based", "FTM": "Trait-Based",
    # Narcissism-Based
    "NPI": "Narcissism-Based", "PNI": "Narcissism-Based", "FFNI": "Narcissism-Based",
    "FFNI_SF": "Narcissism-Based", "NARQ": "Narcissism-Based", "HSNS": "Narcissism-Based",
    "DTM": "Narcissism-Based", "DT4": "Narcissism-Based", "MCMIN": "Narcissism-Based",
    "IPN": "Narcissism-Based",
    # Motivational
    "STBV": "Motivational", "MST": "Motivational", "RFT": "Motivational",
    "SDT": "Motivational", "AAM": "Motivational", "CS": "Motivational",
    # Cognitive
    "PCT": "Cognitive", "SCM": "Cognitive", "CEST": "Cognitive", "FSLS": "Cognitive",
    # Clinical
    "MMPI": "Clinical", "TCI": "Clinical", "TMP": "Clinical", "BDI": "Clinical",
    "GAD7": "Clinical", "SCID": "Clinical", "MCMI": "Clinical",
    "RIT": "Clinical", "TAT": "Clinical", "WAIS": "Clinical",
    # Interpersonal
    "TKI": "Interpersonal", "DISC": "Interpersonal",
    # App/Holistic
    "RIASEC": "App/Holistic", "BT": "App/Holistic", "TEI": "App/Holistic",
    "EM": "App/Holistic", "PAPC": "App/Holistic", "CMOA": "App/Holistic",
}

row_idx = 0
for model_name, (_, count) in model_indices.items():
    for i in range(count):
        df.append({
            'model': model_name,
            'category': CATEGORY_MAP.get(model_name, "Unknown")
        })

df = pd.DataFrame(df)

print(f"\n✅ Metadata:")
print(f"   Rows: {len(df):,}")
print(f"   Models: {df['model'].nunique()}")
print(f"   Categories: {df['category'].nunique()}")
print(f"   Embedding dim: {X.shape[1]}")

## Section 3: Setup Colors & Imports

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Enable Colab plotly rendering
pio.renderers.default = 'colab'

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("⚠️ umap-learn not installed. UMAP projections will be skipped.")

# Category colors (7 categories)
CATEGORY_COLORS = {
    "Trait-Based": "#1f77b4",
    "Narcissism-Based": "#9467bd",
    "Motivational": "#2ca02c",
    "Cognitive": "#ff7f0e",
    "Clinical": "#d62728",
    "Interpersonal": "#e377c2",
    "App/Holistic": "#7f7f7f",
}

# Model colors - 44 distinct colors
MODEL_COLORS = {
    "AAM": "#e377c2", "BDI": "#bcbd22", "BT": "#17becf", "CEST": "#1f77b4",
    "CMOA": "#ff7f0e", "CS": "#2ca02c", "DISC": "#d62728", "DT4": "#9467bd",
    "DTM": "#8c564b", "EM": "#e377c2", "EPM": "#7f7f7f", "FFNI": "#bcbd22",
    "FFNI_SF": "#17becf", "FTM": "#1f77b4", "FSLS": "#ff7f0e", "GAD7": "#2ca02c",
    "HEX": "#d62728", "HSNS": "#9467bd", "IPN": "#8c564b", "MBTI": "#e377c2",
    "MCMI": "#7f7f7f", "MCMIN": "#bcbd22", "MMPI": "#17becf", "MST": "#1f77b4",
    "NARQ": "#ff7f0e", "NPI": "#2ca02c", "OCEAN": "#d62728", "PAPC": "#9467bd",
    "PCT": "#8c564b", "PNI": "#e377c2", "RFT": "#7f7f7f", "RIASEC": "#bcbd22",
    "RIT": "#17becf", "SCID": "#1f77b4", "SCM": "#ff7f0e", "SDT": "#2ca02c",
    "SIXTEENPF": "#d62728", "STBV": "#9467bd", "TAT": "#8c564b", "TCI": "#e377c2",
    "TEI": "#7f7f7f", "TKI": "#bcbd22", "TMP": "#17becf", "WAIS": "#1f77b4",
}

print(f"✅ Imports and colors loaded")
print(f"✅ Plotly renderer: {pio.renderers.default}")
print(f"✅ Data summary: {X.shape[0]:,} embeddings × {X.shape[1]}-dim")
print(f"   Models: {df['model'].nunique()}, Categories: {df['category'].nunique()}")
print(f"   Memory: {X.nbytes / 1e6:.1f} MB")
print(f"\n📊 Category distribution:")
print(df['category'].value_counts().sort_values(ascending=False))

## Section 4: PCA Projection (2D + 3D)

In [ ]:
print("🔄 Computing PCA (50 components)...")
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X).astype(np.float32)

print(f"✅ PCA variance explained (50 components): {100*pca.explained_variance_ratio_.sum():.1f}%")
print(f"   PC1: {100*pca.explained_variance_ratio_[0]:.1f}%")
print(f"   PC2: {100*pca.explained_variance_ratio_[1]:.1f}%")
print(f"   PC3: {100*pca.explained_variance_ratio_[2]:.1f}%")

# 3D PCA visualization
fig = go.Figure()
for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter3d(
        x=X_pca[mask, 0],
        y=X_pca[mask, 1],
        z=X_pca[mask, 2],
        mode='markers',
        name=cat,
        marker=dict(
            size=3,
            color=CATEGORY_COLORS.get(cat, '#999999'),
            opacity=0.7,
            line=dict(width=0),
        ),
        text=[f"<b>{row['model']}</b><br>Category: {row['category']}"
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))

fig.update_layout(
    title='Personality Atlas: PCA 3D Projection (6,694 trait embeddings)',
    scene=dict(
        xaxis_title=f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)",
        yaxis_title=f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)",
        zaxis_title=f"PC3 ({100*pca.explained_variance_ratio_[2]:.1f}%)",
    ),
    height=900,
    template='plotly_dark',
    font=dict(size=12),
    legend=dict(orientation="h", yanchor="top", y=-0.12, xanchor="center", x=0.5, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show(renderer='colab')

print("\n💾 Saving PCA 3D figure...")
fig.write_html('pca_3d.html')
print("✅ Saved to pca_3d.html")

## Section 5: t-SNE Projection

In [ ]:
print("🔄 Computing t-SNE (PCA-50 pre-reduction for speed)...")
pca_50 = PCA(n_components=50)
X_pca_50 = pca_50.fit_transform(X).astype(np.float32)

tsne = TSNE(n_components=2, perplexity=25, learning_rate=10, random_state=42, max_iter=1000, verbose=1)
X_tsne = tsne.fit_transform(X_pca_50).astype(np.float32)
print(f"✅ t-SNE computed (2D)")

# Compute category centroids
centroids = {}
for cat in df['category'].unique():
    mask = df['category'] == cat
    centroids[cat] = X_tsne[mask].mean(axis=0)

# t-SNE 2D visualization
fig = go.Figure()
for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0],
        y=X_tsne[mask, 1],
        mode='markers',
        name=cat,
        marker=dict(
            size=4,
            color=CATEGORY_COLORS.get(cat, '#999999'),
            opacity=0.7,
            line=dict(width=0),
        ),
        text=[f"<b>{row['model']}</b><br>Category: {row['category']}"
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))

# Add category centroids as stars
centroid_names = list(centroids.keys())
centroid_coords = np.array(list(centroids.values()))
fig.add_trace(go.Scatter(
    x=centroid_coords[:, 0],
    y=centroid_coords[:, 1],
    mode='markers+text',
    name='Category centroids',
    marker=dict(size=12, color='white', symbol='star', line=dict(width=2, color='black')),
    text=centroid_names,
    textposition='top center',
    hovertemplate='%{text}<extra></extra>',
    showlegend=True,
))

fig.update_layout(
    title='Personality Atlas: t-SNE 2D Projection (perplexity=25)',
    xaxis_title='t-SNE 1',
    yaxis_title='t-SNE 2',
    height=900,
    template='plotly_dark',
    font=dict(size=12),
    legend=dict(orientation="h", yanchor="top", y=-0.12, xanchor="center", x=0.5, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show(renderer='colab')

print("\n💾 Saving t-SNE figure...")
fig.write_html('tsne_2d.html')
print("✅ Saved to tsne_2d.html")

## Section 6: UMAP Projection (2D)

In [ ]:
from IPython.display import display, clear_output

print("🔄 Computing UMAP (PCA-50 pre-reduction for speed)...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, random_state=42, verbose=False)
X_umap = reducer.fit_transform(X_pca_50).astype(np.float32)
print(f"✅ UMAP computed (2D, neighbors=15)")

fig = go.Figure()
for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter(
        x=X_umap[mask, 0],
        y=X_umap[mask, 1],
        mode='markers',
        name=cat,
        marker=dict(
            size=4,
            color=CATEGORY_COLORS.get(cat, '#999999'),
            opacity=0.7,
            line=dict(width=0),
        ),
        text=[f"<b>{row['model']}</b><br>Category: {row['category']}"
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))

fig.update_layout(
    title='Personality Atlas: UMAP 2D Projection (neighbors=15)',
    xaxis_title='UMAP 1',
    yaxis_title='UMAP 2',
    height=900,
    template='plotly_dark',
    font=dict(size=12),
    legend=dict(orientation="h", yanchor="top", y=-0.12, xanchor="center", x=0.5, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)

print("\n💾 Saving UMAP figure...")
fig.write_html('umap_2d.html')
print("✅ Saved to umap_2d.html")

display(fig)

## Section 7: Per-Category Cluster Analysis

In [ ]:
categories = sorted(df['category'].unique())
n_cats = len(categories)

fig = make_subplots(
    rows=(n_cats + 1) // 2,
    cols=2,
    subplot_titles=categories,
)

for idx, cat in enumerate(categories):
    row = (idx // 2) + 1
    col = (idx % 2) + 1
    mask = df['category'] == cat
    models = sorted(df[mask]['model'].unique())

    for model in models:
        model_mask = mask & (df['model'] == model)
        model_color = MODEL_COLORS.get(model, '#999999')

        fig.add_trace(
            go.Scatter(
                x=X_tsne[model_mask, 0],
                y=X_tsne[model_mask, 1],
                mode='markers',
                name=model,
                marker=dict(size=4, color=model_color, opacity=0.7, line=dict(width=0)),
                showlegend=(idx == 0),
                hovertemplate='<b>%{text}</b><extra></extra>',
                text=[model] * model_mask.sum(),
            ),
            row=row,
            col=col,
        )

fig.update_layout(
    title_text="Per-Category Clustering (t-SNE coordinates, colored by model)",
    height=1200,
    width=1400,
    template='plotly_dark',
    font=dict(size=11),
    showlegend=True,
    legend=dict(x=1.02, y=1.0, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show(renderer='colab')

print("\n💾 Saving per-category analysis...")
fig.write_html('per_category_tsne.html')
print("✅ Saved to per_category_tsne.html")

## Section 8: Category Centroid Similarity Heatmap

In [ ]:
print("🔄 Computing category centroid similarities...")

centroids_embed = {}
for cat in categories:
    mask = df['category'] == cat
    cat_embed = X[mask]
    centroid = cat_embed.mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-10)
    centroids_embed[cat] = centroid

centroids_array = np.array(list(centroids_embed.values()))
sim_matrix = centroids_array @ centroids_array.T

fig = px.imshow(
    sim_matrix,
    x=categories,
    y=categories,
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    labels=dict(color='Cosine<br>Similarity'),
    title='Category Centroid Similarity Matrix (1536-Dim Embedding Space)',
    aspect='auto',
)

fig.update_traces(
    text=np.round(sim_matrix, 3),
    texttemplate='%{text}',
    textfont=dict(size=10),
)

fig.update_layout(
    height=700,
    width=750,
    template='plotly_dark',
    font=dict(size=11),
    xaxis_title='Category',
    yaxis_title='Category',
)
fig.show(renderer='colab')

print("\n💾 Saving heatmap...")
fig.write_html('category_similarity_heatmap.html')
print("✅ Saved to category_similarity_heatmap.html")

print("\n📊 Centroid Similarities (most similar pairs):")
similarities_list = []
for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        if i < j:
            sim = sim_matrix[i, j]
            similarities_list.append((sim, cat1, cat2))

similarities_list.sort(reverse=True)
for sim, cat1, cat2 in similarities_list[:10]:
    print(f"  {cat1:20s} <-> {cat2:20s}: {sim:+.3f}")

## Section 9: Separation Metrics

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

print("🔄 Computing cluster separation and quality metrics...")

labels = df['category'].values
unique_cats = sorted(np.unique(labels))
label_map = {cat: i for i, cat in enumerate(unique_cats)}
label_ints = np.array([label_map[cat] for cat in labels])

# Global metrics (computed on full dataset)
silhouette = silhouette_score(X_pca_50, label_ints)
db_score = davies_bouldin_score(X_pca_50, label_ints)
ch_score = calinski_harabasz_score(X_pca_50, label_ints)

print("\n" + "="*60)
print("GLOBAL CLUSTER QUALITY METRICS")
print("="*60)
print(f"\n  Silhouette Score:        {silhouette:.4f}")
print("  Interpretation: -1 to 1; higher = better separated clusters")
print(f"  Rating: {'EXCELLENT' if silhouette > 0.5 else 'GOOD' if silhouette > 0.3 else 'MODERATE' if silhouette > 0.1 else 'WEAK'}")

print(f"\n  Davies-Bouldin Index:    {db_score:.4f}")
print("  Interpretation: Lower = better (0 = perfect separation)")
print(f"  Rating: {'EXCELLENT' if db_score < 1.0 else 'GOOD' if db_score < 2.0 else 'MODERATE'}")

print(f"\n  Calinski-Harabasz Score: {ch_score:.2f}")
print("  Interpretation: Higher = better cluster definition")

# Per-category silhouette: compute per-sample on full dataset, then average by category
print("\n" + "="*60)
print("PER-CATEGORY MEAN SILHOUETTE SCORES")
print("="*60)
sample_scores = silhouette_samples(X_pca_50, label_ints)
for cat in categories:
    mask = labels == cat
    n_samples = mask.sum()
    mean_score = sample_scores[mask].mean()
    print(f"  {cat:20s} (n={n_samples:4d}): {mean_score:+.4f}")

print("\n" + "="*60)
print("ATLAS COMPOSITION")
print("="*60)
print(f"  Total embeddings:          {len(df):,}")
print(f"  Embedding dimension:       {X.shape[1]} (OpenAI text-embedding-3-small)")
print(f"  Number of models:          {df['model'].nunique()}")
print(f"  Number of categories:      {len(categories)}")
print(f"  Mean embeddings per model: {len(df) / df['model'].nunique():.1f}")

print("\n" + "="*60)
print("EMBEDDINGS BY CATEGORY")
print("="*60)
for cat, count in df['category'].value_counts().sort_values(ascending=False).items():
    pct = 100 * count / len(df)
    print(f"  {cat:20s}: {count:4d} ({pct:5.1f}%)")

## Summary

**Generated Figures:**
- `pca_3d.html` -- PCA 3D visualization (interactive)
- `tsne_2d.html` -- t-SNE 2D global clustering with category centroids
- `umap_2d.html` -- UMAP 2D projection
- `per_category_tsne.html` -- Per-category cluster analysis colored by model
- `category_similarity_heatmap.html` -- 7x7 centroid cosine similarity matrix

**Cluster Quality Metrics:**
- **Silhouette Score**: Measures how well-separated the 7 personality categories are in embedding space
- **Davies-Bouldin Index**: Quantifies cluster compactness and separation (lower = better)
- **Calinski-Harabasz Score**: Evaluates cluster definition quality (higher = better)

**Atlas Details:**
- 44 personality models across 7 categories
- 6,694 trait embeddings at 1536 dimensions (OpenAI text-embedding-3-small)
- Categories: Trait-Based, Narcissism-Based, Motivational, Cognitive, Clinical, Interpersonal, App/Holistic